In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/gzdekzlkaya/wikipedia-text-corpus-for-nlp-and-llm-projects/wikipedia_text_corpus.csv


# Parte 0: Carga del Corpus

**Actividad**
1. Carga el corpus
2. Realiza las etapas de preprocesamiento sobre el corpus

In [2]:
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import string

# 1. Descargar recursos de NLTK necesarios 
nltk.download('punkt')
nltk.download('stopwords')

# --- Actividad 1: Carga del corpus ---
# Ruta para el dataset
file_path = '/kaggle/input/datasets/gzdekzlkaya/wikipedia-text-corpus-for-nlp-and-llm-projects/wikipedia_text_corpus.csv'

# Cargar el CSV en un DataFrame
df = pd.read_csv(file_path)
print(f"Corpus cargado. Dimensiones del dataset: {df.shape}")

# Mostrar las primeras filas para identificar el nombre de la columna que contiene el texto
print("Muestra de los datos originales:")
display(df.head(3))

# --- Actividad 2: Preprocesamiento ---
# NOTA: Asumimos que la columna que contiene los artículos se llama 'text'. 
# Si al imprimir la muestra ves que se llama diferente (ej. 'article', 'content'), cámbialo en la línea de abajo.
columna_texto = 'text' 

# Configurar las stop words en inglés (dado que el corpus de Wikipedia suele estar en este idioma)
stop_words = set(stopwords.words('english'))

def preprocesar_texto(texto):
    # Manejo de valores nulos
    if not isinstance(texto, str):
        return ""
    
    # a. Convertir a minúsculas
    texto = texto.lower()
    
    # b. Tokenización (dividir en palabras)
    tokens = word_tokenize(texto)
    
    # c. Filtrar alfanuméricos (elimina puntuación) y quitar stop words
    tokens_limpios = [word for word in tokens if word.isalnum() and word not in stop_words]
    
    # Reconstruir el texto limpio
    return " ".join(tokens_limpios)

print("\nIniciando preprocesamiento... (Esto puede tomar un par de minutos dependiendo del tamaño del corpus)")

# Aplicar la función de limpieza y guardar en una nueva columna
# Tip: Si el corpus es gigantesco y quieres probar rápido, cambia df[columna_texto] por df[columna_texto].head(1000)
df['texto_limpio'] = df[columna_texto].apply(preprocesar_texto)

print("\nPreprocesamiento completado. Muestra de los datos limpios:")
display(df[[columna_texto, 'texto_limpio']].head(3))

[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Corpus cargado. Dimensiones del dataset: (10859, 2)
Muestra de los datos originales:


,Unnamed: 0,text
0,1,Anovo\n\nAnovo (formerly A Novo) is a computer...
1,2,Battery indicator\n\nA battery indicator (also...
2,3,"Bob Pease\n\nRobert Allen Pease (August 22, 19..."



Iniciando preprocesamiento... (Esto puede tomar un par de minutos dependiendo del tamaño del corpus)

Preprocesamiento completado. Muestra de los datos limpios:


,text,texto_limpio
0,Anovo\n\nAnovo (formerly A Novo) is a computer...,anovo anovo formerly novo computer services co...
1,Battery indicator\n\nA battery indicator (also...,battery indicator battery indicator also known...
2,"Bob Pease\n\nRobert Allen Pease (August 22, 19...",bob pease robert allen pease august 22 1940â j...




# Parte 1: Recuperación con TF-IDF

**Actividad:**

3. Obtén la representación vectorial de los documentos utilizando el modelo TF-IDF
4. A partir de un conjunto de 10 queries, verifica la recuperación del sistema

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# --- Actividad 3: Representación vectorial con TF-IDF ---

# Inicializar el vectorizador TF-IDF
tfidf_vectorizer = TfidfVectorizer()

# Ajustar y transformar la columna de texto limpio en una matriz dispersa
tfidf_matrix = tfidf_vectorizer.fit_transform(df['texto_limpio'])

print(f"Matriz TF-IDF creada. Dimensiones (documentos, vocabulario): {tfidf_matrix.shape}")

Matriz TF-IDF creada. Dimensiones (documentos, vocabulario): (10859, 134063)


In [4]:
# --- Actividad 4: Recuperación a partir de 10 queries ---

# Definimos 10 queries
queries = [
    "python language",
    "linux operating system",
    "artificial intelligence",
    "user interface",
    "computer security",
    "mobile phones",
    "video games",
    "computer hardware",
    "relational database",
    "virtual private network"
]

def recuperar_documentos_tfidf(query, vectorizer, matrix, df_original, top_k=3):
    query_limpia = preprocesar_texto(query)
    query_vec = vectorizer.transform([query_limpia])
    similitudes = cosine_similarity(query_vec, matrix).flatten()
    indices_top = similitudes.argsort()[-top_k:][::-1]
    
    resultados = []
    for idx in indices_top:
        score = similitudes[idx]
        if score > 0: 
            doc_snippet = str(df_original.iloc[idx]['text'])[:100].replace('\n', ' ')
            resultados.append((score, doc_snippet, idx))
            
    return resultados

print("--- Resultados de Recuperación (Top 3 por Query) ---")
for i, query in enumerate(queries, 1):
    print(f"\nQuery {i}: '{query}'")
    resultados = recuperar_documentos_tfidf(query, tfidf_vectorizer, tfidf_matrix, df)
    
    if not resultados:
        print("  -> No se encontraron documentos relevantes.")
    else:
        for rango, (score, snippet, doc_id) in enumerate(resultados, 1):
            print(f"  {rango}. [Score: {score:.4f}] (Doc ID: {doc_id}) -> {snippet}...")

--- Resultados de Recuperación (Top 3 por Query) ---

Query 1: 'python language'
  1. [Score: 0.3122] (Doc ID: 5108) -> Language Server Protocol  The Language Server Protocol (LSP) is an open, JSON-RPC-based protocol for...
  2. [Score: 0.2800] (Doc ID: 9235) -> CircuitPython  CircuitPython is an open source derivative of the MicroPython programming language ta...
  3. [Score: 0.2635] (Doc ID: 6871) -> Doctest  doctest is a module included in the Python programming language's standard library that all...

Query 2: 'linux operating system'
  1. [Score: 0.7593] (Doc ID: 7222) -> Linux adoption  Linux adoption is the adoption of Linux computer operating systems (OS) by household...
  2. [Score: 0.3799] (Doc ID: 10094) -> List of Linux kernel names  Most of the Linux 1.2 and above kernels include a name in the Makefile o...
  3. [Score: 0.3481] (Doc ID: 7067) -> EmperorLinux  EmperorLinux, Inc. is a reseller who, according to "PC Magazine", "specialize in the s...

Query 3: 'artificial int

# Parte 2: Recuperación con BM25

**Actividad:**

5. Implementa un sistema de recuperación usando el modelo BM25.
6. Para el mismo conjunto de 10 queries, verifica la recuperación del sistema

In [5]:
# Librería necesaria para implementar algoritmo Okapi BM25
!pip install rank_bm25

In [6]:
from rank_bm25 import BM25Okapi

# --- Actividad 5: Implementación del sistema BM25 ---
print("Entrenando el modelo BM25...")

# A diferencia de TF-IDF, BM25 requiere que le pasemos una lista de listas de palabras (tokens)
# Por lo tanto, dividimos nuestro texto limpio usando .split()
corpus_tokenizado = [doc.split() for doc in df['texto_limpio']]

# Inicializar el modelo
bm25_model = BM25Okapi(corpus_tokenizado)
print("Modelo BM25 listo.\n")

Entrenando el modelo BM25...
Modelo BM25 listo.



In [7]:
# --- Actividad 6: Recuperación con las mismas 10 queries ---
def recuperar_documentos_bm25(query, modelo_bm25, df_original, top_k=3):
    # Limpiar y tokenizar la query
    query_limpia = preprocesar_texto(query).split()
    
    # Obtener los scores para todos los documentos
    scores = modelo_bm25.get_scores(query_limpia)
    
    # Obtener los índices de los top_k resultados
    indices_top = np.argsort(scores)[-top_k:][::-1]
    
    resultados = []
    for idx in indices_top:
        score = scores[idx]
        if score > 0:
            doc_snippet = str(df_original.iloc[idx]['text'])[:100].replace('\n', ' ')
            resultados.append((score, doc_snippet, idx))
            
    return resultados

print("--- Resultados de Recuperación BM25 (Top 3 por Query) ---")
for i, query in enumerate(queries, 1):
    print(f"\nQuery {i}: '{query}'")
    resultados = recuperar_documentos_bm25(query, bm25_model, df)
    
    if not resultados:
        print("  -> No se encontraron documentos relevantes.")
    else:
        for rango, (score, snippet, doc_id) in enumerate(resultados, 1):
            print(f"  {rango}. [Score: {score:.4f}] (Doc ID: {doc_id}) -> {snippet}...")

--- Resultados de Recuperación BM25 (Top 3 por Query) ---

Query 1: 'python language'
  1. [Score: 17.8047] (Doc ID: 9235) -> CircuitPython  CircuitPython is an open source derivative of the MicroPython programming language ta...
  2. [Score: 16.1090] (Doc ID: 3040) -> MicroPython  MicroPython is a software implementation of the Python 3 programming language, written ...
  3. [Score: 15.4912] (Doc ID: 6871) -> Doctest  doctest is a module included in the Python programming language's standard library that all...

Query 2: 'linux operating system'
  1. [Score: 15.4076] (Doc ID: 7222) -> Linux adoption  Linux adoption is the adoption of Linux computer operating systems (OS) by household...
  2. [Score: 14.3164] (Doc ID: 2645) -> Motorola A780  The Motorola A780 is a cellular PDA running the Linux operating system.  It was intro...
  3. [Score: 13.4847] (Doc ID: 6693) -> Motorola A910  The Motorola A910 is a clamshell mobile phone from Motorola, which uses MontaVista Li...

Query 3: 'arti

# Parte 3: Comparación de resultados

**Actividad:**

7. Verifica cuáles documentos son recuperados (y en qué orden) por cada modelo de recuperación

In [9]:
import pandas as pd

# --- Actividad 7: Comparación de Modelos ---

# Seleccionamos 3 queries donde los modelos difieren
queries_comparacion = [
    "virtual private network",
    "python language",
    "linux operating system"
]

for query in queries_comparacion:
    print(f"BÚSQUEDA: '{query.upper()}'")
    print("-" * 80)
    
    # Recuperar top 3 de cada modelo
    res_tfidf = recuperar_documentos_tfidf(query, tfidf_vectorizer, tfidf_matrix, df)
    res_bm25 = recuperar_documentos_bm25(query, bm25_model, df)
    
    # Formatear para imprimir en columnas
    print(f"{'Ranking':<10} | {'TF-IDF (Doc ID)':<30} | {'BM25 (Doc ID)':<30}")
    print("-" * 80)
    
    for i in range(3):
        # Extraer IDs. Si no hay resultado, poner "N/A"
        id_tfidf = res_tfidf[i][2] if i < len(res_tfidf) else "N/A"
        id_bm25 = res_bm25[i][2] if i < len(res_bm25) else "N/A"
        
        print(f"Top {i+1:<6} | Doc ID: {id_tfidf:<22} | Doc ID: {id_bm25:<22}")
    
    print("\n")

BÚSQUEDA: 'VIRTUAL PRIVATE NETWORK'
--------------------------------------------------------------------------------
Ranking    | TF-IDF (Doc ID)                | BM25 (Doc ID)                 
--------------------------------------------------------------------------------
Top 1      | Doc ID: 5915                   | Doc ID: 838                   
Top 2      | Doc ID: 10010                  | Doc ID: 966                   
Top 3      | Doc ID: 7326                   | Doc ID: 841                   


BÚSQUEDA: 'PYTHON LANGUAGE'
--------------------------------------------------------------------------------
Ranking    | TF-IDF (Doc ID)                | BM25 (Doc ID)                 
--------------------------------------------------------------------------------
Top 1      | Doc ID: 5108                   | Doc ID: 9235                  
Top 2      | Doc ID: 9235                   | Doc ID: 3040                  
Top 3      | Doc ID: 6871                   | Doc ID: 6871             

**Conclusión**

Tras implementar y comparar los modelos TF-IDF y BM25 sobre el mismo corpus, se evidencia una clara evolución en la forma en que los sistemas de recuperación de información interpretan la relevancia. Aunque ambos modelos se basan en el modelo de bolsa de palabras (Bag of Words) y ponderan la rareza de los términos, BM25 demuestra ser un algoritmo significativamente más preciso para entornos reales.

Las principales diferencias observadas se resumen a continuación:

**La saturación de frecuencia:** TF-IDF asume que si una palabra aparece repetidas veces en un documento, su relevancia aumenta de forma casi lineal. Esto provoca fallos críticos (como se observó en la query sobre redes VPN), donde una sola palabra repetitiva cambia el resultado. BM25 soluciona esto aplicando una curva probabilística: entiende que después de cierto número de repeticiones, el término ya alcanzó su "límite de relevancia", permitiendo que otras palabras clave de la búsqueda también tengan peso.

**Normalización por longitud del documento:** TF-IDF tiende a favorecer documentos más largos simplemente porque tienen más espacio para repetir palabras. BM25 incluye un parámetro matemático que penaliza matemáticamente a los documentos muy extensos, asegurando que un artículo corto y conciso sobre el tema tenga una puntuación justa frente a un texto masivo.

**Rendimiento en consultas complejas:** Mientras que TF-IDF se defiende bien en consultas literales y unívocas (conceptos de un solo término muy específico, como nombres de lenguajes de programación), su precisión cae en búsquedas compuestas. BM25 demostró una capacidad superior para equilibrar el peso de múltiples términos, acercándose más a la intención de búsqueda real del usuario.